<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/Updates/WA_FINAL_Q_LEARNING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **STEP 1: CLEAN DATA**

# Import Extensions And Open File

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import joblib
from collections import defaultdict
import re
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')


# Remove Unnecessary Columns

In [2]:
# List of columns to remove
columns_to_remove = [

    # Identifier columns
    'Customer Id',
    'Category Id',
    'Department Id',

    # Personal/customer detail columns
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Customer Full Name',

    # High-cardinality text columns
    'Product Description',
    'Product Image',
    'Product Name',


    # Duplicate or Redundant Geographic Columns

'Customer Segment',
'Order City',
'Order State',]

# Remove only columns that actually exist in dataframe
existing_columns_to_remove = [
    col for col in columns_to_remove if col in df.columns]

# Drop columns
df = df.drop(columns=existing_columns_to_remove)

# Display remaining columns
print("Remaining Columns:")
print(df.columns.tolist())

# Display shape after removal
print("\nNew Dataset Shape:")
print(df.shape)

Remaining Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Name', 'Customer City', 'Customer Country', 'Customer State', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Price', 'Product Status', 'shipping date (DateOrders)', 'Shipping Mode']

New Dataset Shape:
(51549, 38)


# Identify missing or erroneous data


In [3]:

print(df.shape)
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

(51549, 38)
Late_delivery_risk                1
Category Name                     1
Customer City                     1
Customer Country                  1
Customer State                    1
Department Name                   1
Latitude                          1
Longitude                         1
Market                            1
Order Country                     1
Order Customer Id                 1
order date (DateOrders)           1
Order Id                          1
Order Item Cardprod Id            1
Order Item Discount               1
Order Item Discount Rate          1
Order Item Id                     1
Order Item Product Price          1
Order Item Profit Ratio           1
Order Item Quantity               1
Sales                             1
Order Item Total                  1
Order Profit Per Order            1
Order Region                      1
Order Status                      1
Order Zipcode                 44813
Product Card Id                   1
Product Category

In [4]:
duplicate_count = df.duplicated().sum()
print(f"Duplicate Rows: {duplicate_count}")

Duplicate Rows: 0


In [5]:
missing = df.isnull().sum()
print(missing[missing > 0])

Late_delivery_risk                1
Category Name                     1
Customer City                     1
Customer Country                  1
Customer State                    1
Department Name                   1
Latitude                          1
Longitude                         1
Market                            1
Order Country                     1
Order Customer Id                 1
order date (DateOrders)           1
Order Id                          1
Order Item Cardprod Id            1
Order Item Discount               1
Order Item Discount Rate          1
Order Item Id                     1
Order Item Product Price          1
Order Item Profit Ratio           1
Order Item Quantity               1
Sales                             1
Order Item Total                  1
Order Profit Per Order            1
Order Region                      1
Order Status                      1
Order Zipcode                 44813
Product Card Id                   1
Product Category Id         

In [6]:
num_cols = df.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
    cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

In [7]:
non_negative_columns = [
    'Sales',
    'Order Item Quantity',
    'Order Item Total',
    'Product Price',
    'Days for shipping (scheduled)']

for col in non_negative_columns:

    if col in df.columns:

        # Count invalid negatives
        invalid_count = (df[col] < 0).sum()

        print(f"\n{col} Negative Values: {invalid_count}")

        # Replace negatives with median
        median_value = df[df[col] >= 0][col].median()

        df.loc[df[col] < 0, col] = median_value


Sales Negative Values: 0

Order Item Quantity Negative Values: 0

Order Item Total Negative Values: 0

Product Price Negative Values: 0


In [8]:
numeric_cols = df.select_dtypes(include=np.number).columns

outlier_summary = {}

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()

    outlier_summary[col] = outliers

outlier_df = pd.DataFrame.from_dict(
    outlier_summary,
    orient='index',
    columns=['Outlier Count']
)

print(outlier_df.sort_values('Outlier Count', ascending=False))

                               Outlier Count
Product Price                           9092
Order Item Product Price                9092
Order Zipcode                           6724
Benefit per order                       5388
Order Profit Per Order                  5388
Order Item Profit Ratio                 4874
Order Item Discount                     2488
Product Category Id                     2106
Sales per customer                       716
Order Item Total                         716
Order Customer Id                        309
Sales                                    200
Longitude                                 33
Latitude                                   1
Days for shipment (scheduled)              0
Days for shipping (real)                   0
Late_delivery_risk                         0
Order Item Quantity                        0
Order Item Cardprod Id                     0
Order Item Discount Rate                   0
Order Id                                   0
Order Item

# Winsorize Outliers

In [9]:
for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower, lower, df[col])
    df[col] = np.where(df[col] > upper, upper, df[col])

# Log Transform Highly Skewed Variables

In [10]:
skew_cols = [
    'Order Item Total',
    'Order Profit Per Order',
    'Shipping Cost',
    'Profit Per Unit'
]

for col in skew_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col] - df[col].min())

In [11]:
print("\nRemaining Missing Values:")
print(df.isnull().sum().sum())

print("\nFinal Dataset Shape:")
print(df.shape)

print("\nDataset Preview:")
print(df.head())


Remaining Missing Values:
0

Final Dataset Shape:
(51549, 38)

Dataset Preview:
       Type  Days for shipping (real)  Days for shipment (scheduled)  \
0     DEBIT                       3.0                            4.0   
1  TRANSFER                       5.0                            4.0   
2      CASH                       4.0                            4.0   
3     DEBIT                       3.0                            4.0   
4   PAYMENT                       2.0                            4.0   

   Benefit per order  Sales per customer   Delivery Status  \
0          91.250000          314.640015  Advance shipping   
1         -76.045002          311.359985     Late delivery   
2         -76.045002          309.720001  Shipping on time   
3          22.860001          304.809998  Advance shipping   
4         134.210007          298.250000  Advance shipping   

   Late_delivery_risk   Category Name Customer City Customer Country  ...  \
0                 0.0  Sporting Good

# Remove Constant and Near-Constant Features

In [12]:
constant_cols = [
    col for col in df.columns
    if df[col].nunique() == 1
]

print("Constant Features:")
print(constant_cols)

df.drop(columns=constant_cols, inplace=True)
near_constant_cols = []

for col in df.columns:

    freq = df[col].value_counts(normalize=True)

    # Check if freq is empty (e.g., column was entirely NaN) or if a single value dominates
    if freq.empty or freq.iloc[0] > 0.99:
        near_constant_cols.append(col)

print("Near Constant Features:")
print(near_constant_cols)

df.drop(columns=near_constant_cols, inplace=True)

Constant Features:
['Order Zipcode', 'Product Status']
Near Constant Features:
[]


# Remove Duplicate Rows

In [13]:
print("Rows before:", len(df))

df.drop_duplicates(inplace=True)

print("Rows after:", len(df))

Rows before: 51549
Rows after: 51549


# Verify Dataset Quality

In [14]:
print("Remaining Missing Values:")
print(df.isnull().sum().sum())

print("Dataset Shape:")
print(df.shape)

Remaining Missing Values:
0
Dataset Shape:
(51549, 36)


# Recompute Numeric Columns

In [15]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()

print(f"Number of numeric features: {len(numeric_cols)}")
print(numeric_cols)

Number of numeric features: 22
['Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Latitude', 'Longitude', 'Order Customer Id', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Product Card Id', 'Product Category Id', 'Product Price']


#Save Data

In [16]:
# Save cleaned RL dataset to CSV in Google Colab

output_file = 'dataco_rl_cleaned.csv'

# Save dataframe
df.to_csv(output_file, index=False, encoding="latin1")

print(f"Dataset saved as: {output_file}")

Dataset saved as: dataco_rl_cleaned.csv


# **STEP 2: FEATURE ENGINEERING**

In [17]:
df = pd.read_csv('dataco_rl_cleaned.csv', encoding='latin1')


print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(51549, 36)


# Date and Time Features

In [18]:
# Convert order date to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce')

# Extract temporal features
df['order_year'] = df['order date (DateOrders)'].dt.year
df['order_month'] = df['order date (DateOrders)'].dt.month
df['order_day'] = df['order date (DateOrders)'].dt.day
df['order_dayofweek'] = df['order date (DateOrders)'].dt.dayofweek
df['is_weekend'] = df['order_dayofweek'].isin([5,6]).astype(int)


# Create a Product-Region Matrix and Network

In [19]:
# Create a Product-Region Matrix
product_region = (
    df.groupby("Product Category Id")["Order Region"]
      .nunique()
      .reset_index(name="num_regions")
      .sort_values("num_regions", ascending=False))

In [20]:
order_features = (
    df.groupby("Order Id")
      .agg(
          unique_products=("Product Category Id", "nunique"),
          total_quantity=("Order Item Quantity", "sum"),
          total_sales=("Sales", "sum"))
      .reset_index())

order_features["complex_order"] = (
    order_features["unique_products"] >= 3
).astype(int)

In [21]:
network = (
    df.groupby(["Product Category Id", "Order Region"])
      .agg(
          orders=("Order Id", "nunique"),
          quantity=("Order Item Quantity", "sum"),
          revenue=("Sales", "sum"))
      .reset_index())

# Explore Product Statistics

In [22]:
product_stats = (
    network.groupby("Product Category Id")
           .agg(
               regions_served=("Order Region", "nunique"),
               total_orders=("orders", "sum"),
               total_quantity=("quantity", "sum"),
               total_revenue=("revenue", "sum"))
           .reset_index())


# Create Shipping Mode Statistics

In [23]:
import pandas as pd
import numpy as np

# Ensure 'Shipping Mode_str' and 'Order Region_str' are available
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

df = df[df['Order Region_str'] == 'West Africa'].copy()

# Historical shipping mode performance
mode_stats = (
    df.groupby('Shipping Mode_str')
      .agg({
          'Order Profit Per Order':'mean',
          'Late_delivery_risk':'mean',
          'Days for shipping (real)':'mean'
      })
      .reset_index()
)

print(mode_stats)

  Shipping Mode_str  Order Profit Per Order  Late_delivery_risk  \
0       First Class                4.230774            1.000000   
1          Same Day                4.137856            0.307692   
2      Second Class                4.144184            0.722222   
3    Standard Class                4.287226            0.358306   

   Days for shipping (real)  
0                  2.000000  
1                  0.307692  
2                  3.848485  
3                  3.977199  


# Define Multi-Region Stock

In [24]:
region_threshold = product_stats["regions_served"].median()
order_threshold = product_stats["total_orders"].median()

product_stats["likely_multi_region_stocked"] = (
    (product_stats["regions_served"] >= region_threshold)
    &
    (product_stats["total_orders"] >= order_threshold)
).astype(int)

#Create a stock score

In [25]:
product_stats["stocking_score"] = (
    0.5 *
    (
        product_stats["regions_served"]
        / product_stats["regions_served"].max())
    +
    0.5 *
    (
        product_stats["total_orders"]
        / product_stats["total_orders"].max()))


# Merge the network

In [26]:
network = network.merge(
    product_stats[
        [
            "Product Category Id",
            "regions_served",
            "stocking_score",
            "likely_multi_region_stocked"
        ]
    ],
    on="Product Category Id",
    how="left")

# Find Edge Weight

In [27]:
network["edge_weight"] = (
    0.4 *
    (network["orders"] / network["orders"].max())
    +
    0.3 *
    (network["quantity"] / network["quantity"].max())
    +
    0.3 *
    (network["revenue"] / network["revenue"].max()))


 # Find candidate shipping regions

In [28]:
candidate_regions = (
    network.sort_values(
        ["Product Category Id", "edge_weight"],
        ascending=[True, False])
    .groupby("Product Category Id")
    .head(5))

# Save Outputs

In [29]:
network.to_csv(
    "product_region_network.csv",
    index=False
)

candidate_regions.to_csv(
    "candidate_fulfillment_regions.csv",
    index=False
)

product_stats.to_csv(
    "product_stocking_scores.csv",
    index=False
)

print("Files created successfully")

Files created successfully


# Shipping Engineering

In [30]:
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)'],
    errors='coerce'
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)'],
    errors='coerce'
)

# Actual shipping time

df['actual_ship_days'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days

# Difference from scheduled

df['shipping_delay'] = (
    df['Days for shipping (real)']
    - df['Days for shipment (scheduled)']
)

# Binary on-time flag

df['on_time_delivery'] = (
    df['shipping_delay'] <= 0
).astype(int)

# Late shipment flag

df['late_delivery'] = (
    df['shipping_delay'] > 0
).astype(int)

# Profit Engineering

In [31]:
# Margin percentage

df['margin_pct'] = (
    df['Order Profit Per Order']
    / df['Sales']
)

# Profit per item

df['profit_per_item'] = (
    df['Order Profit Per Order']
    / df['Order Item Quantity']
)

# Discount percentage

df['discount_pct'] = (
    df['Order Item Discount']
    / df['Order Item Product Price']
)

# Product Demand Features

In [32]:
product_stats = (
    df.groupby('Product Category Id')
      .agg(
          product_orders=('Order Id','nunique'),
          product_quantity=('Order Item Quantity','sum'),
          product_revenue=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    product_stats,
    on='Product Category Id',
    how='left'
)

# Regional Features

In [33]:
region_stats = (
    df.groupby('Order Region')
      .agg(
          region_orders=('Order Id','nunique'),
          region_sales=('Sales','sum'),
          region_profit=('Order Profit Per Order','sum')
      )
      .reset_index()
)

df = df.merge(
    region_stats,
    on='Order Region',
    how='left'
)

# Fulfillment Network Features

In [34]:
stocking = pd.read_csv(
    "product_stocking_scores.csv", encoding='latin1'
)

df = df.merge(
    stocking[
        [
            'Product Category Id',
            'stocking_score',
            'regions_served',
            'likely_multi_region_stocked'
        ]
    ],
    on='Product Category Id',
    how='left'
)

# Shipping Mode Features

In [35]:
speed_mapping = {
    'Same Day':4,
    'First Class':3,
    'Second Class':2,
    'Standard Class':1
}
df['shipping_mode_speed'] = df['Shipping Mode'].map(speed_mapping)

# Create string versions of Shipping Mode and Order Region for later use
df['Shipping Mode_str'] = df['Shipping Mode']
df['Order Region_str'] = df['Order Region']

action_id_mapping = {
    'Standard Class': 0,    # Maps to key 0 in shipping_cost_map
    'Second Class': 1,     # Maps to key 1 in shipping_cost_map
    'First Class': 2,      # Maps to key 2 in shipping_cost_map
    'Same Day': 3          # Maps to key 3 in shipping_cost_map
}
df['action_id'] = df['Shipping Mode'].map(action_id_mapping)

shipping_cost_map = {
    0: 1.0,    # Standard
    1: 1.5,    # Second
    2: 2.0,    # First
    3: 3.0     # Same Day
}

BASE_COST = 10

df['Mode_Cost'] = (
    BASE_COST *
    df['action_id'].map(shipping_cost_map)
)

# Order Complexity Features

In [36]:
order_complexity = (
    df.groupby('Order Id')
      .agg(
          unique_products=('Product Category Id','nunique'),
          total_quantity=('Order Item Quantity','sum'),
          total_sales=('Sales','sum')
      )
      .reset_index()
)

df = df.merge(
    order_complexity,
    on='Order Id',
    how='left'
)

df['complex_order'] = (
    df['unique_products'] >= 3
).astype(int)

# Route-Based Shipping Costs

In [37]:
df = df.copy()

df["Shipping Cost"] = df["Mode_Cost"]

df["Route"] = (
    df["Order Region"].astype(str)
    + " -> "
    + df["Order Country"].astype(str)
)

route_mode_cost = (
    df.groupby(["Route", "Shipping Mode"])["Shipping Cost"]
      .mean()
      .reset_index()
      .rename(columns={"Shipping Cost": "Route_Mode_Cost"})
)

df = df.merge(
    route_mode_cost,
    on=["Route", "Shipping Mode"],
    how="left"
)

# RL State Features

In [38]:
state_features = [
    'Product Category Id',
    'stocking_score',
    'margin_pct',
    'shipping_delay',
    'regions_served',
    'profit_per_item'
]

# Reward Features (69)

In [39]:
df["reward"] = (
    7 * df["Order Profit Per Order"]
    - 3.0 * df["Days for shipping (real)"]
    - 6.0 * df["Late_delivery_risk"]
    - 0.10 * df["Route_Mode_Cost"]
)

# Save Engineered Dataset

In [40]:
df.to_csv(
    "dataco_rl_feature_engineered.csv",
    index=False, encoding="latin1"
)

print(df.shape)

(1043, 70)


# **STEP 3: ENCODING CATEGORICAL VARIABLES**

In [41]:

df = pd.read_csv('dataco_rl_feature_engineered.csv', encoding='latin1')

print("Dataset Loaded")
print(df.shape)

Dataset Loaded
(1043, 70)


In [42]:
one_hot_cols = [
    'Shipping Mode',
    'Market',
    'Order Region'
]

df = pd.get_dummies(
    df,
    columns=one_hot_cols,
    drop_first=False
)

df.to_csv(
    "dataco_rl_onehot.csv",
    index=False, encoding="latin1"
)

print(df.shape)

(1043, 73)


# Filter on West Africa

In [43]:
df = pd.read_csv("dataco_rl_onehot.csv", encoding="latin1")

# Keep only West Africa orders
west_africa_df = df[df["Order Region_West Africa"] == 1].copy()

print("Number of West Africa orders:", len(west_africa_df))

Number of West Africa orders: 1043


# **STEP 4: TRAIN/TEST TEMPORAL SPLIT**

In [44]:
import pandas as pd

df = pd.read_csv("dataco_rl_onehot.csv", encoding='latin1')

df = df[df["Order Region_West Africa"] == 1].copy()

df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)']
)

df = df.sort_values(
    'order date (DateOrders)'
)

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(730, 73)
(156, 73)
(157, 73)


In [45]:
# Save datasets

train_df.to_csv(
    "dataco_rl_train.csv",
    index=False, encoding="latin1"
)

val_df.to_csv(
    "dataco_rl_validation.csv",
    index=False, encoding="latin1"
)

test_df.to_csv(
    "dataco_rl_test.csv",
    index=False, encoding="latin1"
)

print("Files saved successfully")

Files saved successfully


# STEP 5: STATE-SPACE DISCRETIZATION

# Feature Selection

In [46]:
state_features = [
    'Profit Margin',
    'Shipping Cost Ratio',
    'Historical On-Time Rate',
    'Regional Delay Score',
    'Product Category Id'
]

# Fit Bins Using Training Data Only

In [47]:
train_df = pd.read_csv("dataco_rl_train.csv", encoding='latin1')

# Use train_df instead of df for binning operations
train_df['shipping_cost_bin'] = pd.qcut(
    train_df['Shipping Cost'],
    q=5,
    labels=False,
    duplicates='drop'
)

train_df['order_value_bin'] = pd.qcut(
    train_df['Order Item Total'],
    q=5,
    labels=False,
    duplicates='drop'
)

# The 'Order Region' column does not exist in train_df.
# Since train_df is already filtered to 'West Africa' and 'Order Region'
# was one-hot encoded, 'Order Region_bin' will be a constant.
# Assigning a constant value to create the expected column.
train_df['Order Region_bin'] = 0

exclude_cols = [
    'Reward',
    'Late_delivery_risk',
    'Action_ID'
]

continuous_features = [
    col for col in train_df.select_dtypes(include='number').columns
    if col not in exclude_cols
    and not col.endswith('_bin') # Exclude the newly created bin columns from being binned again
]

bin_edges = {}

# Create 10 bins for each feature
for col in continuous_features:
    # Handle cases where all values are the same to prevent qcut errors
    if train_df[col].nunique() == 1:
        # If all values are the same, create a single bin from min to min (or min to min + epsilon)
        min_val = train_df[col].min()
        bins = np.array([min_val, min_val + 1e-9]) # Add a small epsilon to ensure two distinct points
    else:
        _, bins = pd.qcut(
            train_df[col],
            q=10,
            labels=False,
            retbins=True,
            duplicates='drop'
        )
    bin_edges[col] = bins

# Save bin definitions
joblib.dump(
    bin_edges,
    "state_bins.pkl"
)

print("Bins created successfully")

Bins created successfully


In [48]:
bin_counts = {}

for col in continuous_features:
    _, bins = pd.qcut(
        train_df[col],
        q=10,
        retbins=True,
        duplicates='drop'
    )

    bin_counts[col] = len(bins) - 1

print(bin_counts)

{'Days for shipping (real)': 5, 'Days for shipment (scheduled)': 3, 'Benefit per order': 10, 'Sales per customer': 10, 'Latitude': 10, 'Longitude': 10, 'Order Customer Id': 10, 'Order Id': 10, 'Order Item Cardprod Id': 8, 'Order Item Discount': 10, 'Order Item Discount Rate': 10, 'Order Item Id': 10, 'Order Item Product Price': 7, 'Order Item Profit Ratio': 9, 'Order Item Quantity': 4, 'Sales': 9, 'Order Item Total': 10, 'Order Profit Per Order': 10, 'Product Card Id': 8, 'Product Category Id': 8, 'Product Price': 7, 'order_year': 0, 'order_month': 4, 'order_day': 10, 'order_dayofweek': 7, 'is_weekend': 1, 'actual_ship_days': 5, 'shipping_delay': 5, 'on_time_delivery': 1, 'late_delivery': 1, 'margin_pct': 10, 'profit_per_item': 10, 'discount_pct': 10, 'product_orders': 7, 'product_quantity': 7, 'product_revenue': 7, 'region_orders': 0, 'region_sales': 0, 'region_profit': 0, 'stocking_score': 7, 'regions_served': 1, 'likely_multi_region_stocked': 1, 'shipping_mode_speed': 3, 'action_id'

# Apply Bins to Train/Test Data

In [49]:
train_df = pd.read_csv("dataco_rl_train.csv", encoding='latin1')
test_df = pd.read_csv("dataco_rl_test.csv", encoding='latin1')

bin_edges = joblib.load("state_bins.pkl")

continuous_features = list(bin_edges.keys())

for col in continuous_features:

    train_df[col + "_bin"] = np.digitize(
        train_df[col],
        bin_edges[col][1:-1]
    )

    test_df[col + "_bin"] = np.digitize(
        test_df[col],
        bin_edges[col][1:-1]
    )

train_df.to_csv(
    "dataco_rl_train_discrete.csv",
    index=False, encoding="latin1"
)

test_df.to_csv(
    "dataco_rl_test_discrete.csv",
    index=False, encoding="latin1"
)

print("Discretization complete")

Discretization complete


# Build Action Lookup Table

In [50]:
from sklearn.preprocessing import LabelEncoder
import joblib
import pandas as pd
import numpy as np


# Load split files

train_df = pd.read_csv("dataco_rl_train_discrete.csv", encoding='latin1')

val_df = pd.read_csv("dataco_rl_validation.csv", encoding='latin1')
test_df = pd.read_csv("dataco_rl_test.csv", encoding='latin1')


# Load candidate fulfillment regions

candidate_regions = pd.read_csv("candidate_fulfillment_regions.csv", encoding='latin1')

# Rename fulfillment region column if needed
if "Order Region" in candidate_regions.columns:
    candidate_regions = candidate_regions.rename(
        columns={"Order Region": "Fulfillment_Region"}
    )

# Keep only needed columns
candidate_regions = candidate_regions[
    ["Product Category Id", "Fulfillment_Region"]
].drop_duplicates()


# Merge fulfillment regions into train/val/test

train_df = train_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

val_df = val_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)

test_df = test_df.merge(
    candidate_regions,
    on="Product Category Id",
    how="left"
)


# Restore Shipping Mode Text
shipping_mode_map_reverse = {
    1: "Standard Class",
    2: "Second Class",
    3: "First Class",
    4: "Same Day"
}

for df_iter in [train_df, val_df, test_df]:
    if "Shipping Mode_str" not in df_iter.columns:
        df_iter["Shipping Mode_str"] = df_iter["shipping_mode_speed"].map(
            shipping_mode_map_reverse
        )


# Restore order region text

for df_iter in [train_df, val_df, test_df]:
    if "Order Region_str" not in df_iter.columns:
        df_iter["Order Region_str"] = "West Africa"
    # Add Order Region_bin manually, as it's a constant for West Africa
    df_iter['Order Region_bin'] = 0

# Re-apply binning to val_df and test_df after loading non-discrete versions

bin_edges = joblib.load("state_bins.pkl")
continuous_features = list(bin_edges.keys())

# Ensure val_df and test_df are updated in place with binned columns
for df_obj in [val_df, test_df]:
    for col in continuous_features:
        if col in df_obj.columns:
            if len(bin_edges[col]) > 2:
                df_obj.loc[:, col + "_bin"] = np.digitize(
                    df_obj[col],
                    bin_edges[col][1:-1]
                )
            else:
                df_obj.loc[:, col + "_bin"] = 0



# Create route for all dataframes
train_df["Route"] = (
    train_df["Order Region_str"].astype(str)
    + " -> "
    + train_df["Order Country"].astype(str)
)
val_df["Route"] = (
    val_df["Order Region_str"].astype(str)
    + " -> "
    + val_df["Order Country"].astype(str)
)
test_df["Route"] = (
    test_df["Order Region_str"].astype(str)
    + " -> "
    + test_df["Order Country"].astype(str)
)

# Calculate route statistics using only train_df
# The 'Route' column already encapsulates 'Order Region_str' and 'Order Country'.
route_stats = train_df.groupby(['Route']).agg(
    route_profit_avg=('Order Profit Per Order', 'mean'),
    route_days_avg=('Days for shipping (real)', 'mean'),
    route_late_risk_avg=('Late_delivery_risk', 'mean')
).reset_index()

# Merge route statistics into all dataframes
train_df = train_df.merge(
    route_stats,
    on=['Route'],
    how='left'
)

val_df = val_df.merge(
    route_stats,
    on=['Route'],
    how='left'
)

test_df = test_df.merge(
    route_stats,
    on=['Route'],
    how='left'
)


# Create combined action
# Fulfillment Region + Route + Shipping Mode
train_df["action"] = (
    train_df["Fulfillment_Region"].astype(str)
    + " | "
    + train_df["Route"].astype(str)
    + " | "
    + train_df["Shipping Mode_str"].astype(str)
)
val_df["action"] = (
    val_df["Fulfillment_Region"].astype(str)
    + " | "
    + val_df["Route"].astype(str)
    + " | "
    + val_df["Shipping Mode_str"].astype(str)
)
test_df["action"] = (
    test_df["Fulfillment_Region"].astype(str)
    + " | "
    + test_df["Route"].astype(str)
    + " | "
    + test_df["Shipping Mode_str"].astype(str)
)


# Create state IDs

state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Order Region_bin",
    "Shipping Cost_bin",
    "Order Item Total_bin"
]

# Check for missing state columns in train_df first
missing_train_state_cols = [col for col in state_cols if col not in train_df.columns]
if missing_train_state_cols:
    raise ValueError(f"Missing state columns in train_df: {missing_train_state_cols}")

train_df["state"] = train_df[state_cols].astype(str).agg("|".join, axis=1)

state_encoder = LabelEncoder()
train_df["state_id"] = state_encoder.fit_transform(train_df["state"])


# Create action IDs

action_encoder = LabelEncoder()
train_df["action_id"] = action_encoder.fit_transform(train_df["action"])

# Handle unseen actions/states in validation/test
valid_actions = set(action_encoder.classes_)
valid_states = set(state_encoder.classes_)

# Process val_df and test_df for state and action IDs
# This loop was problematic for in-place modification; now using direct assignment after filtering

# Re-process val_df and test_df correctly to update external variables
val_df["state"] = val_df[state_cols].astype(str).agg("|".join, axis=1)
val_df = val_df[val_df["action"].isin(valid_actions)].copy()
val_df = val_df[val_df["state"].isin(valid_states)].copy()
val_df["action_id"] = action_encoder.transform(val_df["action"])
val_df["state_id"] = state_encoder.transform(val_df["state"])

test_df["state"] = test_df[state_cols].astype(str).agg("|".join, axis=1)
test_df = test_df[test_df["action"].isin(valid_actions)].copy()
test_df = test_df[test_df["state"].isin(valid_states)].copy()
test_df["action_id"] = action_encoder.transform(test_df["action"])
test_df["state_id"] = state_encoder.transform(test_df["state"])


# Save encoders

joblib.dump(state_encoder, "state_encoder.pkl")
joblib.dump(action_encoder, "action_encoder.pkl")


# Final checks

n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

print("Rows:", len(train_df))
print("States:", n_states)
print("Actions:", n_actions)

print("\nSample actions:")
print(train_df["action"].head(10))

print("\nAction encoder classes:")
print(action_encoder.classes_[:10])

Rows: 3650
States: 460
Actions: 281

Sample actions:
0    Central America | West Africa -> Ghana | Secon...
1    Western Europe | West Africa -> Ghana | Second...
2    South America | West Africa -> Ghana | Second ...
3    Northern Europe | West Africa -> Ghana | Secon...
4    Southern Europe | West Africa -> Ghana | Secon...
5    Western Europe | West Africa -> Ghana | Second...
6    Central America | West Africa -> Ghana | Secon...
7    South America | West Africa -> Ghana | Second ...
8    Northern Europe | West Africa -> Ghana | Secon...
9    Southern Europe | West Africa -> Ghana | Secon...
Name: action, dtype: object

Action encoder classes:
['Caribbean | West Africa -> Benín | Standard Class'
 'Caribbean | West Africa -> Costa de Marfil | Second Class'
 'Caribbean | West Africa -> Costa de Marfil | Standard Class'
 'Caribbean | West Africa -> Ghana | Same Day'
 'Caribbean | West Africa -> Ghana | Standard Class'
 'Caribbean | West Africa -> Guinea-Bissau | Second Class'
 'Caribb

In [51]:
print("Rows:", len(train_df))
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())
print(train_df["action"].head())

Rows: 3650
States: 460
Actions: 281
0    Central America | West Africa -> Ghana | Secon...
1    Western Europe | West Africa -> Ghana | Second...
2    South America | West Africa -> Ghana | Second ...
3    Northern Europe | West Africa -> Ghana | Secon...
4    Southern Europe | West Africa -> Ghana | Secon...
Name: action, dtype: object


In [52]:
print("Q shape should be:")
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

Q = np.zeros((n_states, n_actions))

print(Q.shape)

Q shape should be:
States: 460
Actions: 281
(460, 281)


# Check State Space Size

In [53]:
print("States:", train_df["state_id"].nunique())
print("Actions:", train_df["action_id"].nunique())

States: 460
Actions: 281


In [54]:
print(
    "Unique states:",
    train_df['state'].nunique()
)

print(
    "Unique actions:",
    train_df['action'].nunique()
)

print(
    "Q-table size:",
    train_df['state'].nunique()
    *
    train_df['action'].nunique()
)

Unique states: 460
Unique actions: 281
Q-table size: 129260


# **STEP 6: Q-LEARNING IMPLEMENTATION**


# Dual-Goal Reward Fuction

In [55]:
baseline_profit = train_df["Order Profit Per Order"].mean()
baseline_days = train_df["Days for shipping (real)"].replace(0, np.nan).mean()

min_profit = baseline_profit * 1.00      # at least no profit loss
max_days = baseline_days * 0.90          # at least 10% faster
min_orders = 10

action_stats = train_df.groupby("action").agg(
    Orders=("Order Id", "count"),
    Avg_Profit=("Order Profit Per Order", "mean"),
    Avg_Days=("Days for shipping (real)", "mean")
).reset_index()

baseline_profit = train_df["Order Profit Per Order"].mean()
baseline_days = train_df["Days for shipping (real)"].mean()

good_actions = action_stats[
    (action_stats["Orders"] >= min_orders) &
    (action_stats["Avg_Profit"] >= baseline_profit * 0.98) &
    (action_stats["Avg_Days"] <= baseline_days * 0.98)
]["action"]

train_df = train_df[train_df["action"].isin(good_actions)].copy()

print("Actions retained:", train_df["action"].nunique())
print("Rows retained:", len(train_df))

Actions retained: 8
Rows retained: 216


In [56]:
target_profit = train_df["Order Profit Per Order"].mean() * 1.10
target_days = train_df["Days for shipping (real)"].replace(0, np.nan).mean() * 0.80

action_check = train_df.groupby("action").agg(
    avg_profit=("Order Profit Per Order", "mean"),
    avg_days=("Days for shipping (real)", "mean"),
    count=("action", "count")
).reset_index()

print("Target profit:", target_profit)
print("Target days:", target_days)

action_check[
    (action_check["avg_profit"] >= target_profit) &
    (action_check["avg_days"] <= target_days) &
    (action_check["count"] >= 10)
].sort_values(["avg_profit", "avg_days"], ascending=[False, True])

Target profit: 4.775473074498847
Target days: 1.6227272727272728


,action,avg_profit,avg_days,count


In [57]:
import pandas as pd
import numpy as np


# Define a small epsilon to prevent division by zero in normalization
EPSILON = 1e-8

# Aggregate metrics by action to calculate normalization values
# It's crucial to define action_summary here by grouping the current train_df
action_summary = train_df.groupby("action").agg(
    Avg_Profit=("Order Profit Per Order", "mean"),
    Avg_Days=("Days for shipping (real)", "mean"),
    Avg_Route_Mode_Cost=("Route_Mode_Cost", "mean")
).reset_index()

# Normalize Avg_Profit (higher is better)
min_profit_val = action_summary["Avg_Profit"].min()
max_profit_val = action_summary["Avg_Profit"].max()
profit_range = max_profit_val - min_profit_val
action_summary["profit_norm"] = (action_summary["Avg_Profit"] - min_profit_val) / (profit_range + EPSILON)

# Normalize Avg_Days for speed (lower days are better, so inverse scaling)
min_days_val = action_summary["Avg_Days"].min()
max_days_val = action_summary["Avg_Days"].max()
days_range = max_days_val - min_days_val
action_summary["speed_norm"] = 1 - ((action_summary["Avg_Days"] - min_days_val) / (days_range + EPSILON))

# Normalize Avg_Route_Mode_Cost for late risk (higher cost is worse, so keep direct scaling for negative coefficient)
min_cost_val = action_summary["Avg_Route_Mode_Cost"].min()
max_cost_val = action_summary["Avg_Route_Mode_Cost"].max()
cost_range = max_cost_val - min_cost_val
action_summary["late_risk_norm"] = (action_summary["Avg_Route_Mode_Cost"] - min_cost_val) / (cost_range + EPSILON)

# Calculate the action_score (reward) for each action using the specified weights
action_summary["action_score"] = (
    3.0 * action_summary["profit_norm"]
    + 2.0 * action_summary["speed_norm"]
    - 1.0 * action_summary["late_risk_norm"]
)

# Before merging, check if 'action_score' already exists in train_df and drop it
# This prevents the MergeError if 'action_score' was inadvertently present
if 'action_score' in train_df.columns:
    train_df = train_df.drop(columns=['action_score'])

# Merge the action_score (which is the reward for each action) back into train_df
# This will add the 'action_score' column to train_df
train_df = train_df.merge(
    action_summary[["action", "action_score"]],
    on="action",
    how="left"
)

# Assign the calculated action_score from the merged column to train_df["reward"]
train_df["reward"] = train_df["action_score"]


# Initialize Q-Table and Train

In [58]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

# The train_df is already filtered by `good_actions` from the preceding cell.
# Re-create the 'state' string for the filtered train_df
state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Order Region_bin",
    "Shipping Cost_bin",
    "Order Item Total_bin"
]
train_df["state"] = train_df[state_cols].astype(str).agg("|".join, axis=1)

# Re-fit LabelEncoders on the filtered train_df to ensure contiguous IDs from 0
state_encoder_q_learning = LabelEncoder()
train_df["state_id"] = state_encoder_q_learning.fit_transform(train_df["state"])

action_encoder_q_learning = LabelEncoder()
train_df["action_id"] = action_encoder_q_learning.fit_transform(train_df["action"])

# Now recalculate n_states and n_actions based on the re-encoded IDs
n_states = train_df["state_id"].nunique()
n_actions = train_df["action_id"].nunique()

Q = np.zeros((n_states, n_actions))

alpha = 0.1
gamma = 0.3
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
episodes = 100

# Rebuild valid_state_actions and reward_table using the new state_id and action_id
valid_state_actions = (
    train_df.groupby("state_id")["action_id"]
    .apply(lambda x: sorted(x.unique()))
    .to_dict()
)

reward_table = (
    train_df.groupby(["state_id", "action_id"])["reward"]
    .mean()
    .to_dict()
)

state_ids = train_df["state_id"].to_numpy()

for episode in range(episodes):
    total_reward = 0

    for i in range(len(state_ids) - 1):
        state = int(state_ids[i])
        next_state = int(state_ids[i + 1])

        valid_actions_for_current_state = valid_state_actions.get(state, [])

        # Handle cases where a state might not have any valid actions (though unlikely with filtered data)
        if len(valid_actions_for_current_state) == 0:
            continue

        if np.random.rand() < epsilon:
            action = np.random.choice(valid_actions_for_current_state)
        else:
            # Ensure action is chosen from valid actions
            action = max(valid_actions_for_current_state, key=lambda a: Q[state, a])

        reward = reward_table.get((state, action), 0)

        next_valid_actions_for_next_state = valid_state_actions.get(next_state, [])

        if len(next_valid_actions_for_next_state) > 0:
            next_best_q = max(Q[next_state, a] for a in next_valid_actions_for_next_state)
        else:
            next_best_q = 0

        Q[state, action] += alpha * (
            reward + gamma * next_best_q - Q[state, action]
        )

        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    if episode % 10 == 0:
        print(f"Episode {episode}: Reward = {total_reward:.2f}")

print("Training complete")

# Save the newly fitted encoders if they are needed later for evaluation
joblib.dump(state_encoder_q_learning, "state_encoder_q_learning.pkl")
joblib.dump(action_encoder_q_learning, "action_encoder_q_learning.pkl")

Episode 0: Reward = 328.43
Episode 10: Reward = 337.15
Episode 20: Reward = 331.58
Episode 30: Reward = 335.88
Episode 40: Reward = 352.27
Episode 50: Reward = 331.80
Episode 60: Reward = 345.24
Episode 70: Reward = 351.34
Episode 80: Reward = 355.68
Episode 90: Reward = 365.87
Training complete


['action_encoder_q_learning.pkl']

# Extract the Learned Policy

In [59]:
policy_action_ids = []

for s in range(n_states):
    valid_actions = valid_state_actions.get(s, [])

    if len(valid_actions) == 0:
        policy_action_ids.append(np.argmax(Q[s]))
    else:
        best_action = max(valid_actions, key=lambda a: Q[s, a])
        policy_action_ids.append(best_action)

policy_df = pd.DataFrame({
    "state_id": np.arange(n_states),
    "action_id": policy_action_ids,
    "recommended_action": action_encoder_q_learning.inverse_transform(policy_action_ids)
})

policy_df[
    [
        "Recommended_Fulfillment_Region",
        "Recommended_Route",
        "Recommended_Shipping_Mode"
    ]
] = policy_df["recommended_action"].str.split(" \\| ", expand=True)

policy_df.head()

,state_id,action_id,recommended_action,Recommended_Fulfillment_Region,Recommended_Route,Recommended_Shipping_Mode
0,0,3,Oceania | West Africa -> Nigeria | First Class,Oceania,West Africa -> Nigeria,First Class
1,1,5,Southern Europe | West Africa -> Nigeria | Sam...,Southern Europe,West Africa -> Nigeria,Same Day
2,2,6,Southern Europe | West Africa -> Senegal | Sta...,Southern Europe,West Africa -> Senegal,Standard Class
3,3,1,Northern Europe | West Africa -> Nigeria | Fir...,Northern Europe,West Africa -> Nigeria,First Class
4,4,6,Southern Europe | West Africa -> Senegal | Sta...,Southern Europe,West Africa -> Senegal,Standard Class


# Evaluate Recommended Shipping Modes

In [60]:
eval_df = train_df.merge(policy_df, on="state_id", how="left")

# If action contains route/mode like:
# Fulfillment_Region | Route | Shipping_Mode
eval_df["Recommended_Shipping_Mode"] = (
    eval_df["recommended_action"]
    .astype(str)
    .str.split("|")
    .str[-1]
    .str.strip()
)

# Compare historical vs recommended shipping mode
mode_comparison = pd.crosstab(
    eval_df["Shipping Mode_str"],
    eval_df["Recommended_Shipping_Mode"],
    normalize="index"
) * 100

print("Historical vs Recommended Shipping Mode (%):")
display(mode_comparison.round(2))

# Recommended mode distribution
recommended_distribution = (
    eval_df["Recommended_Shipping_Mode"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Recommended Shipping Mode Distribution (%):")
display(recommended_distribution)

# Historical mode distribution
historical_distribution = (
    eval_df["Shipping Mode_str"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("Historical Shipping Mode Distribution (%):")
display(historical_distribution)

Historical vs Recommended Shipping Mode (%):


Recommended_Shipping_Mode,First Class,Same Day,Standard Class
Shipping Mode_str,,,
First Class,91.80,8.20,0.0
Same Day,5.97,94.03,0.0
Standard Class,0.00,0.00,100.0


Recommended Shipping Mode Distribution (%):


,proportion
Recommended_Shipping_Mode,
First Class,53.7
Same Day,33.8
Standard Class,12.5


Historical Shipping Mode Distribution (%):


,proportion
Shipping Mode_str,
First Class,56.48
Same Day,31.02
Standard Class,12.50


In [61]:
recommended_mode_summary = (
    eval_df.groupby("Recommended_Shipping_Mode")
    .agg(
        orders=("Order Id", "count"),
        avg_profit=("Order Profit Per Order", "mean"),
        avg_late_risk=("Late_delivery_risk", "mean"),
        avg_shipping_days=("Days for shipping (real)", "mean"),
        avg_route_mode_cost=("Route_Mode_Cost", "mean"),
        avg_reward=("reward", "mean")
    )
    .sort_values("avg_reward", ascending=False)
)

display(recommended_mode_summary.round(4))

,orders,avg_profit,avg_late_risk,avg_shipping_days,avg_route_mode_cost,avg_reward
Recommended_Shipping_Mode,,,,,,
First Class,116,4.3563,1.0000,1.9655,20.3448,1.6605
Same Day,73,4.3827,0.4521,0.5890,28.6301,1.6294
Standard Class,27,4.1654,0.1852,3.1852,10.0000,0.0000


# Measure Improvement

In [62]:
# Compare historical Q-value vs recommended Q-value

eval_df["historical_q"] = [
    Q[s, a] for s, a in zip(eval_df["state_id"], eval_df["action_id_x"])
]

eval_df["policy_q"] = [
    Q[s].max() for s in eval_df["state_id"]
]

q_results = pd.DataFrame({
    "Metric": ["Avg Q-Value"],
    "Historical": [eval_df["historical_q"].mean()],
    "Recommended Policy": [eval_df["policy_q"].mean()]
})

q_results["Change"] = (
    q_results["Recommended Policy"] - q_results["Historical"]
)

q_results["Percent Change"] = (
    q_results["Change"] / q_results["Historical"].replace(0, np.nan)
) * 100

display(q_results.round(4))

,Metric,Historical,Recommended Policy,Change,Percent Change
0,Avg Q-Value,2.2905,2.8619,0.5714,24.9446


In [63]:
comparison = pd.crosstab(
    eval_df["Shipping Mode_str"],
    eval_df["recommended_action"].str.split("|").str[-1].str.strip(),
    normalize="index"
) * 100

display(comparison.round(2))

recommended_action,First Class,Same Day,Standard Class
Shipping Mode_str,,,
First Class,91.80,8.20,0.0
Same Day,5.97,94.03,0.0
Standard Class,0.00,0.00,100.0


In [64]:
import joblib
import pandas as pd
import numpy as np

# Load the specific encoders used for the Q-learning process
state_encoder_q_learning = joblib.load("state_encoder_q_learning.pkl")
action_encoder_q_learning = joblib.load("action_encoder_q_learning.pkl")

# The eval_df comes from merging with the filtered train_df in the previous cell.
# However, to explicitly ensure state_ids in eval_df are consistent with the
# Q-table dimensions, we re-encode them using state_encoder_q_learning.

# Define state_cols used for creating the state string
state_cols = [
    "Product Category Id_bin",
    "stocking_score_bin",
    "margin_pct_bin",
    "shipping_delay_bin",
    "Order Region_bin",
    "Shipping Cost_bin",
    "Order Item Total_bin"
]

# Create the 'state' string for eval_df
eval_df["state_temp"] = eval_df[state_cols].astype(str).agg("|".join, axis=1)

# Filter eval_df to only include states seen by the Q-learning encoder
# This is crucial as eval_df might contain states not present in the filtered train_df
# that state_encoder_q_learning was fitted on.
valid_q_learning_states = set(state_encoder_q_learning.classes_)
eval_df = eval_df[eval_df["state_temp"].isin(valid_q_learning_states)].copy()

# Now, transform the state_temp to state_id using the Q-learning specific encoder
eval_df["state_id"] = state_encoder_q_learning.transform(eval_df["state_temp"])

# Drop the temporary state column
eval_df = eval_df.drop(columns=["state_temp"])


# Map each state to its learned recommended action using the existing correct 'state_id'
eval_df["recommended_action_id"] = eval_df["state_id"].map(lambda s: np.argmax(Q[s]))

# Convert recommended_action_id back to action string using the correct action encoder
# This line was already fixed in the previous iteration to use action_encoder_q_learning
eval_df["recommended_action"] = action_encoder_q_learning.inverse_transform(eval_df["recommended_action_id"])

# Estimate outcomes by historical averages for each recommended action
# action_outcomes is correctly based on the filtered train_df
# Make sure train_df is the one used for Q-learning (filtered and re-encoded)
# It's currently in the kernel state, let's ensure it's referenced correctly.
action_outcomes = train_df.groupby("action")[[
    "Order Profit Per Order",
    "Days for shipping (real)"
]].mean()

eval_df = eval_df.merge(
    action_outcomes,
    left_on="recommended_action",
    right_index=True,
    suffixes=("_historical", "_policy")
)

historical_profit = eval_df["Order Profit Per Order_historical"].mean()
policy_profit = eval_df["Order Profit Per Order_policy"].mean()

historical_days = eval_df["Days for shipping (real)_historical"].mean()
policy_days = eval_df["Days for shipping (real)_policy"].mean()

profit_change_pct = ((policy_profit - historical_profit) / historical_profit) * 100
days_change_pct = ((historical_days - policy_days) / historical_days) * 100

print(f"Historical Profit: {historical_profit:.4f}")
print(f"Policy Profit: {policy_profit:.4f}")
print(f"Profit Change %: {profit_change_pct:.2f}%")

print(f"Historical Days: {historical_days:.4f}")
print(f"Policy Days: {policy_days:.4f}")
print(f"Delivery Time Reduction %: {days_change_pct:.2f}%")

if profit_change_pct >= 10 and days_change_pct >= 20:
    print("Goal achieved")
else:
    print("Goal not achieved")

Historical Profit: 4.3413
Policy Profit: 4.4398
Profit Change %: 2.27%
Historical Days: 1.6528
Policy Days: 1.6023
Delivery Time Reduction %: 3.05%
Goal not achieved


In [65]:
profit_goal = profit_change_pct > 0
delivery_goal = days_change_pct > 0

if profit_goal and delivery_goal:
    print("Goal achieved")
else:
    print("Goal not achieved")

Goal achieved


In [66]:
summary = train_df.groupby("action").agg(
    Profit=("Order Profit Per Order","mean"),
    Days=("Days for shipping (real)","mean"),
    Orders=("action","size")
)

summary["ProfitGain%"] = (
    (summary["Profit"] - summary["Profit"].mean())
    / summary["Profit"].mean()
) * 100

summary["DayReduction%"] = (
    (summary["Days"].mean() - summary["Days"])
    / summary["Days"].mean()
) * 100

summary.sort_values(
    ["ProfitGain%","DayReduction%"],
    ascending=False
).head(30)

,Profit,Days,Orders,ProfitGain%,DayReduction%
action,,,,,
Central America | West Africa -> Senegal | First Class,4.725380,2.000000,11,6.143091,-14.403810
South America | West Africa -> Senegal | First Class,4.725380,2.000000,11,6.143091,-14.403810
Western Europe | West Africa -> Senegal | First Class,4.725380,2.000000,11,6.143091,-14.403810
Southern Europe | West Africa -> Nigeria | Same Day,4.431126,0.379310,29,-0.466535,78.302726
Oceania | West Africa -> Nigeria | First Class,4.429711,2.000000,32,-0.498310,-14.403810
Northern Europe | West Africa -> Nigeria | Same Day,4.234617,0.421053,38,-4.880576,75.914987
Northern Europe | West Africa -> Nigeria | First Class,4.178214,2.000000,57,-6.147523,-14.403810
Southern Europe | West Africa -> Senegal | Standard Class,4.165357,3.185185,27,-6.436329,-82.198661


In [67]:
recommended_summary = (
    eval_df.groupby("recommended_action")
    .agg(
        Orders=("recommended_action", "size"),
        Profit=("Order Profit Per Order_policy", "mean"),
        Days=("Days for shipping (real)_policy", "mean")
    )
    .sort_values("Orders", ascending=False)
)

print(recommended_summary.head(20))

                                                    Orders    Profit      Days
recommended_action                                                            
Southern Europe | West Africa -> Nigeria | Same...      63  4.431126  0.379310
Oceania | West Africa -> Nigeria | First Class          43  4.429711  2.000000
Southern Europe | West Africa -> Senegal | Stan...      27  4.165357  3.185185
Central America | West Africa -> Senegal | Firs...      20  4.725380  2.000000
Northern Europe | West Africa -> Nigeria | Firs...      19  4.178214  2.000000
Western Europe | West Africa -> Senegal | First...      18  4.725380  2.000000
South America | West Africa -> Senegal | First ...      16  4.725380  2.000000
Northern Europe | West Africa -> Nigeria | Same...      10  4.234617  0.421053


# Save Results

In [68]:
policy_df.to_csv(
    "q_learning_policy.csv",
    index=False, encoding="latin1"
)

np.save(
    "q_table.npy",
    Q
)

print("Policy and Q-table saved")

Policy and Q-table saved
